In [2]:
from pathlib import Path

EOD_PATH = Path("data") / "EOD_STGO"

In [3]:
from chiricoca.config import setup_style
setup_style(dpi=150)

In [4]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import geopandas as gpd
import networkx as nx

In [ ]:
import huedhued.eod_scl as eod

viajes = eod.read_trips(EOD_PATH)

# descartamos sectores que no sean relevantes en los orígenes y destinos de los viajes
viajes = viajes[
    (viajes["SectorOrigen"] != "Exterior a RM")
    & (viajes["SectorDestino"] != "Exterior a RM")
    & (viajes["SectorOrigen"] != "Extensión Sur-Poniente")
    & (viajes["SectorDestino"] != "Extensión Sur-Poniente")
    & pd.notnull(viajes["SectorOrigen"])
    & pd.notnull(viajes["SectorDestino"])
]

personas = eod.read_people(EOD_PATH)

viajes_persona = viajes.merge(personas)

viajes_persona["Peso"] = (
    viajes_persona["FactorExpansion"] * viajes_persona["FactorPersona"]
)

print(
    "{} viajes expandidos a {}".format(
        len(viajes_persona), int(viajes_persona["Peso"].sum())
    )
)

In [ ]:
matriz = (
    viajes_persona.groupby(["ComunaOrigen", "ComunaDestino"])
    .agg(n_viajes=("Peso", "sum"))
    .reset_index()
)

matriz.head()

In [ ]:
from chiricoca.base.weights import normalize_rows

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    matriz.set_index(["ComunaOrigen", "ComunaDestino"])["n_viajes"]
    .unstack(fill_value=0)
    .pipe(normalize_rows)
    ,
    cmap="inferno_r",
    linewidth=1,
)

In [ ]:
orden_comunas = (viajes_persona[['ComunaOrigen', 'SectorOrigen']]
                 .drop_duplicates()
                 .sort_values('SectorOrigen')
                 ['ComunaOrigen']
                 .values
)

orden_comunas

In [ ]:
full_matrix = (
    matriz.set_index(["ComunaOrigen", "ComunaDestino"])["n_viajes"]
    .unstack(fill_value=0)
    .pipe(normalize_rows)
).loc[orden_comunas, orden_comunas]

sns.heatmap(full_matrix, cmap="inferno_r", linewidth=1)

In [ ]:
propositos = ['Al trabajo', 'Al estudio', 'De compras', 'Trámites', 'Buscar o Dejar a alguien', 'Visitar a alguien', 'Recreación', 'De salud']
viajes_persona['Proposito'].isin(propositos).mean()

In [ ]:
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(18,8))


for i, (ax, prop) in enumerate(zip(axes.flatten(), propositos)):
    prop_matrix = (viajes_persona[viajes_persona['Proposito'] == prop]
                   .drop_duplicates('Persona', keep='first')
                   .groupby(['ComunaOrigen', 'ComunaDestino'])
        ['Peso'].sum()
        .unstack(fill_value=0)
        ).join(full_matrix[[]])
    
    for col in full_matrix.columns:
        if not col in prop_matrix.columns:
            prop_matrix[col] = 0.0

    prop_matrix = prop_matrix.loc[full_matrix.index,full_matrix.columns]

    sns.heatmap(
        prop_matrix.pipe(normalize_rows),
        cmap="inferno_r",
        linewidth=1,
        cbar=True,
        ax=ax,
        xticklabels=False,
        yticklabels=False
    )

    ax.set_title(prop)
    ax.set_xlabel('')
    ax.set_ylabel('')

    ax.set_aspect('equal')

    if i % 4 == 0:
        ax.set_yticks(np.arange(len(prop_matrix)) + 0.5, prop_matrix.index.values, fontsize='x-small')

    #if i >= 4:
    ax.set_xticks(np.arange(len(prop_matrix)) + 0.5, prop_matrix.columns.values, rotation=90, fontsize='x-small')


In [ ]:
matrices_zonas = {}

for prop in propositos:
    matrices_zonas[prop] = (
        viajes_persona[
            (viajes_persona["Proposito"] == prop)
            & (viajes_persona["ZonaOrigen"])
            & (viajes_persona["ZonaDestino"])
            & (viajes_persona["ZonaOrigen"] != viajes_persona["ZonaDestino"])
        ]
        .groupby(["ComunaOrigen", "ZonaOrigen", "ZonaDestino"])
        # ojo: deberíamos usar la distancia ponderada
        .agg(n_viajes=("Peso", "sum"), distancia=("DistManhattan", "mean"))
        .sort_values("n_viajes", ascending=False)
        .assign(cumsum_viajes=lambda x: x["n_viajes"].cumsum())
        .assign(cumsum_viajes=lambda x: x["cumsum_viajes"] / x["cumsum_viajes"].max())
        .reset_index()
    )

matrices_zonas["Al trabajo"]

In [ ]:
sns.heatmap(
    matrices_zonas["Al trabajo"]
    .set_index(["ZonaOrigen", "ZonaDestino"])["n_viajes"]
    .unstack(fill_value=0).pipe(normalize_rows),
    cmap="inferno_r",
    linewidth=0,
)

In [ ]:
from chiricoca.networks.stats import summary
od_network = nx.from_pandas_edgelist(
    matrices_zonas["Al trabajo"],
    source="ZonaOrigen",
    target="ZonaDestino",
    edge_attr=["n_viajes", 'distancia'],
    create_using=nx.DiGraph,
)

summary(od_network)

In [74]:
pos = nx.forceatlas2_layout(od_network)

In [ ]:
nx.draw_networkx(od_network, pos=pos, with_labels=False, node_size=10, arrows=False)

In [76]:
zones = eod.read_zone_design(EOD_PATH)

In [ ]:
from chiricoca.geo.utils import clip_area_geodataframe

scl_bounds = [-70.88006218, -33.67612715, -70.43015094, -33.31069169]

scl_zones = clip_area_geodataframe(zones.to_crs('epsg:4326'), scl_bounds).to_crs(zones.crs)
scl_zones.plot(column='Comuna')

In [ ]:
geo_pos = scl_zones.set_index('ID').centroid.map(lambda x: (x.x, x.y)).to_dict()
geo_pos

In [79]:
od_network.remove_nodes_from(set(zones['ID']) - set(scl_zones['ID']))

In [ ]:
from cytoolz import valfilter
disconnected_nodes = valfilter(lambda x: x == 0, dict(nx.degree(od_network)))
print(disconnected_nodes)
if disconnected_nodes:
    od_network.remove_nodes_from(disconnected_nodes.keys())

In [ ]:
from chiricoca.geo.figures import figure_from_geodataframe

fig, ax = figure_from_geodataframe(scl_zones) 
scl_zones.plot(ax=ax, facecolor='none', edgecolor='#abacab')

nx.draw_networkx(od_network, pos=geo_pos, with_labels=False, node_size=10, arrows=False, ax=ax)


In [ ]:
edge_n_viajes = pd.Series(map(lambda x: x[2]["n_viajes"], od_network.edges(data=True))).rename('n_viajes')
edge_n_viajes.describe()

In [ ]:
edge_distancia = pd.Series(map(lambda x: x[2]["distancia"], od_network.edges(data=True))).rename('distancia')
edge_distancia.describe()

In [138]:
nx.set_edge_attributes(
    od_network,
    {
        (x[0], x[1]): (x[2]["n_viajes"] + 1) / (x[2]["distancia"] + 1)
        for x in od_network.edges(data=True)
    },
    "intensidad_de_uso",
)

In [149]:
nx.set_edge_attributes(
    od_network,
    {
        (x[0], x[1]): np.sqrt(x[2]["n_viajes"]) * np.sqrt(x[2]["distancia"] + 1)
        for x in od_network.edges(data=True)
    },
    "flujo_sqrt",
)

In [ ]:
edge_centrality_dist = pd.Series(nx.edge_betweenness_centrality(od_network, weight='distancia')).rename('centrality_(distancia)')
edge_centrality_dist

In [ ]:
edge_centrality_int = pd.Series(nx.edge_betweenness_centrality(od_network, weight='intensidad_de_uso')).rename('centrality_(intensidad)')
edge_centrality_int

In [ ]:
edge_centrality_flux = pd.Series(nx.edge_betweenness_centrality(od_network, weight='flujo_sqrt')).rename('centrality_(flux)')
edge_centrality_flux

In [ ]:
from chiricoca.geo.figures import small_multiples_from_geodataframe

fig, axes = small_multiples_from_geodataframe(scl_zones, 5, height=5, col_wrap=3)

for (ax, df) in zip(axes, [edge_n_viajes, edge_distancia, edge_centrality_dist, edge_centrality_int, edge_centrality_flux]):
    edge_df_norm = df
    edge_df_norm = edge_df_norm / edge_df_norm.sum() * 100
    nx.draw_networkx_edges(
        od_network,
        pos=geo_pos,
        arrows=False,
        width=edge_df_norm,
        alpha=0.8,
        ax=ax
    )
    ax.set_title(df.name)

In [341]:
def prepare_geo_network(matrix_df):
    od_network = nx.from_pandas_edgelist(
        matrix_df,
        source="ZonaOrigen",
        target="ZonaDestino",
        edge_attr=["n_viajes", "distancia"],
        create_using=nx.DiGraph,
    )

    od_network.remove_nodes_from(set(zones["ID"]) - set(scl_zones["ID"]))

    disconnected_nodes = valfilter(lambda x: x == 0, dict(nx.degree(od_network)))

    if disconnected_nodes:
        od_network.remove_nodes_from(disconnected_nodes.keys())

    nx.set_edge_attributes(
        od_network,
        {
            (x[0], x[1]): np.sqrt((x[2]["n_viajes"] + 1) * (x[2]["distancia"] + 1))
            for x in od_network.edges(data=True)
        },
        "flujo_sqrt",
    )

    nx.set_edge_attributes(
        od_network,
        nx.edge_betweenness_centrality(od_network, weight="intensidad_de_uso"),
        name="centrality_(intensidad)",
    )

    nx.set_edge_attributes(
        od_network,
        nx.edge_betweenness_centrality(od_network, weight="flujo_sqrt"),
        name="centrality_(flux)",
    )

    return od_network

In [ ]:
viajes_persona['Proposito'].value_counts()

In [ ]:
matrices_sexo = (
    viajes_persona[
        (~viajes_persona["Proposito"].isin(['volver a casa', 'Al estudio', 'Por estudio', 'Recreación', 'Comer o Tomar algo', 'Por trabajo']))
        & (viajes_persona["ZonaOrigen"] != viajes_persona["ZonaDestino"])
        & (pd.isnull(viajes_persona['DondeEstudia']))
        & (viajes_persona['AnoNac'].between(1953, 1994))
    ]
    .groupby(["Sexo", "ZonaOrigen", "ZonaDestino"])
    # ojo: deberíamos usar la distancia ponderada
    .agg(n_viajes=("Peso", "sum"), distancia=("DistManhattan", "mean"))
    .sort_values("n_viajes", ascending=False)
    .sort_values(['Sexo', 'n_viajes'])
    .pipe(lambda x: x[x['n_viajes'] >= 10])
)

matrices_sexo

In [ ]:
od_hombres = prepare_geo_network(matrices_sexo.loc['Hombre'].reset_index())
summary(od_hombres)

In [ ]:
od_mujeres = prepare_geo_network(matrices_sexo.loc['Mujer'].reset_index())
summary(od_mujeres)

In [415]:
od_sexo = {'hombres': od_hombres, 'mujeres': od_mujeres}

In [ ]:
fig, axes = small_multiples_from_geodataframe(scl_zones, 2, height=4)

for ax, _sex in zip(axes, od_sexo.keys()):
    _od = od_sexo[_sex]
    edge_df_norm = pd.Series(nx.get_edge_attributes(_od, 'centrality_(flux)'))
    edge_df_norm = edge_df_norm / edge_df_norm.sum() * 100
    nx.draw_networkx_edges(
        _od,
        pos=geo_pos,
        arrows=False,
        width=edge_df_norm,
        alpha=0.8,
        ax=ax
    )
    ax.set_title(f'{_sex}')

In [420]:
for _od in od_sexo.values():
    nx.set_node_attributes(_od, nx.load_centrality(_od, weight='n_viajes'), 'node_centrality')

In [ ]:
fig, axes = small_multiples_from_geodataframe(scl_zones, 2, height=4, col_wrap=2)

for ax, _sex in zip(axes, od_sexo.keys()):
    _od = od_sexo[_sex]
    edge_df_norm = pd.Series(nx.get_edge_attributes(_od, 'centrality_(flux)'))
    edge_df_norm = edge_df_norm / edge_df_norm.sum() * 100
    nx.draw_networkx_edges(
        _od,
        pos=geo_pos,
        arrows=False,
        width=edge_df_norm,
        alpha=0.8,
        ax=ax
    )

    node_df_norm = pd.Series(nx.get_node_attributes(_od, 'node_centrality'))
    node_df_norm = node_df_norm / node_df_norm.sum() * 500
    nx.draw_networkx_nodes(_od, pos=geo_pos, node_size=node_df_norm, ax=ax, node_color='white', edgecolors='black', linewidths=0.5)
    ax.set_title(f'{_sex}')

In [ ]:
from chiricoca.base.weights import variance_stabilization

flux_diff = (
    pd.Series(
        nx.get_edge_attributes(od_sexo["hombres"], "centrality_(flux)"), name="flux_h"
    )
    .to_frame()
    .join(
        pd.Series(
            nx.get_edge_attributes(od_sexo["mujeres"], "centrality_(flux)"),
            name="flux_m",
        ),
        how="outer",
    )
    .pipe(np.sqrt)
    .fillna(0)
    .add(0.000001)
    .pipe(variance_stabilization)
    .drop('flux_h', axis=1)
    .rename({'flux_m': 'diff_flux'}, axis=1)
    #.assign(diff_flux=lambda x: x["flux_m"] - x["flux_h"])
)

flux_diff.describe()

In [431]:
flux_threshold = 0.05

In [ ]:
from itertools import chain

edges_m = set(flux_diff[flux_diff['diff_flux'] > flux_threshold].index.values)
nodes_m = set(chain(*edges_m))
subset_od_m = nx.subgraph_view(od_sexo['mujeres'], filter_node=lambda x: x in nodes_m, filter_edge=lambda x,y: (x,y) in edges_m)
summary(subset_od_m)

In [ ]:
edges_h = set(flux_diff[flux_diff['diff_flux'] < -flux_threshold].index.values)
nodes_h = set(chain(*edges_h))
subset_od_h = nx.subgraph_view(od_sexo['hombres'], filter_node=lambda x: x in nodes_h, filter_edge=lambda x,y: (x,y) in edges_h)
summary(subset_od_h)

In [434]:
od_subsets = {'hombres': subset_od_h, 'mujeres': subset_od_m}

In [435]:
if not (Path("data") / "scl_network_basemap.tif").exists():
    import contextily as cx

    bounds = scl_zones.buffer(1000).to_crs('epsg:4326').total_bounds

    scl_img, scl_ext = cx.bounds2raster(
        bounds[0],
        bounds[1],
        bounds[2],
        bounds[3],
        Path("data") / "scl_network_basemap.tif",
        ll=True,
        source=cx.providers.CartoDB.DarkMatterNoLabels,
        zoom=12,
    )

In [436]:
from chiricoca.maps.utils import add_basemap

In [ ]:
node_df_norm / node_df_norm.sum() * 1000

In [ ]:
fig, axes = small_multiples_from_geodataframe(scl_zones, 2, height=7, col_wrap=2)

_centr = 'centrality_(flux)'
colors = ('orange', 'green')
for ax, _sex, color in zip(axes, od_subsets.keys(), colors):
    add_basemap(ax, Path("data") / "scl_network_basemap.tif", scl_zones)

    _od = od_subsets[_sex]
    edge_df_norm = pd.Series(nx.get_edge_attributes(_od, _centr))
    edge_df_norm = edge_df_norm / edge_df_norm.sum() * 150
    nx.draw_networkx_edges(
        _od,
        pos=geo_pos,
        arrows=False,
        width=edge_df_norm,
        alpha=1.0,
        ax=ax,
        edge_color=color
    )

    node_df_norm = pd.Series(nx.get_node_attributes(_od, 'node_centrality'))
    node_df_norm = node_df_norm / node_df_norm.sum() * 3000
    nx.draw_networkx_nodes(_od, pos=geo_pos, node_size=node_df_norm, ax=ax, node_color='white', edgecolors='black', linewidths=0.5)
    ax.set_title(f'{_sex}')